In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

In [2]:
# Cell 1

import gc
import json
import math
import os
import shutil
import time
import warnings
from collections import defaultdict
from contextlib import nullcontext
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
torch.set_num_threads(1)
torch.backends.cudnn.benchmark = True
if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
    torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("medium")

# ---------------------------------------------------------------------
# USER FLAGS
# ---------------------------------------------------------------------
RUN_TRAIN = True
RUN_FINAL_TRAIN = True
RUN_INFER = True
DEBUG = False
FORCE_SINGLE_GPU = True
VAL_WINDOWS_PER_MONTH = 48
TOPK_CKPTS = 3


class CFG:
    ROOT = (
        "/kaggle/input/competitions/"
        "anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/"
        "aisehack-theme-2"
    )
    RAW_ROOT = f"{ROOT}/raw"
    TEST_ROOT = f"{ROOT}/test_in"
    LAT_LON_PATH = f"{RAW_ROOT}/lat_long.npy"

    OUT_DIR = "/kaggle/working/aisehack_deadline"
    CKPT_DIR = f"{OUT_DIR}/checkpoints"
    SELECT_CKPT_DIR = f"{CKPT_DIR}/select"
    FINAL_CKPT_DIR = f"{CKPT_DIR}/final"
    SELECT_LOG_PATH = f"{OUT_DIR}/select_log.json"
    FINAL_LOG_PATH = f"{OUT_DIR}/final_log.json"
    SELECT_STATS_PATH = f"{OUT_DIR}/select_stats.json"
    SELECT_HOTSPOT_PATH = f"{OUT_DIR}/select_hotspot_prior.npy"
    STATS_PATH = f"{OUT_DIR}/stats.json"
    HOTSPOT_PATH = f"{OUT_DIR}/hotspot_prior.npy"
    PREDS_PATH = "/kaggle/working/preds.npy"

    MONTHS = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]

    H = 140
    W = 124
    PAD_H = 144
    PAD_W = 128
    TIME_IN = 10
    TIME_OUT = 16
    HORIZON = TIME_IN + TIME_OUT
    STRIDE = 1

    BASE_FEATURES = [
        "cpm25",
        "q2",
        "t2",
        "u10",
        "v10",
        "swdown",
        "pblh",
        "psfc",
        "rain",
        "PM25",
        "NH3",
        "SO2",
        "NOx",
        "NMVOC_e",
        "NMVOC_finn",
        "bio",
    ]
    DERIVED_FEATURES = [
        "wind_speed",
        "wind_divergence",
        "ventilation_index",
        "pblh_inverse",
    ]
    DYNAMIC_FEATURES = BASE_FEATURES + DERIVED_FEATURES
    STATIC_FEATURES = [
        "lat",
        "lon",
        "cpm25_mean_10",
        "cpm25_trend_3",
        "cpm25_std_10",
        "hotspot_prior",
    ]

    TARGET_IDX = 0
    N_DYNAMIC = len(DYNAMIC_FEATURES)
    N_STATIC = len(STATIC_FEATURES)

    EMISSION_FEATURES = {
        "PM25",
        "NH3",
        "SO2",
        "NOx",
        "NMVOC_e",
        "NMVOC_finn",
        "bio",
    }
    WIND_TANH_FEATURES = {"u10", "v10"}

    EPS = 1e-6
    EPISODE_THRESHOLD_STD = 1.5
    HOTSPOT_KERNEL = 5
    HOTSPOT_WEIGHT_SCALE = 1.5

    MODEL_BASE = 48
    MODEL_HIDDEN = 160
    DROPOUT = 0.10

    TRAIN_BATCH = 4
    GRAD_ACCUM = 2
    INFER_BATCH = 4
    NUM_WORKERS = 0
    EMA_DECAY = 0.995

    SEED = 42
    EPOCHS = 10
    PATIENCE = 2
    WARMUP_EPOCHS = 1
    FINAL_SEEDS = [42, 52]

    LR = 8e-4
    WEIGHT_DECAY = 1e-4

    LOSS_WEIGHTS = {
        "delta": 0.35,
        "global_smape": 0.25,
        "episode_smape": 0.20,
        "episode_under": 0.10,
        "hotspot": 0.10,
    }


os.makedirs(CFG.OUT_DIR, exist_ok=True)
os.makedirs(CFG.CKPT_DIR, exist_ok=True)
os.makedirs(CFG.SELECT_CKPT_DIR, exist_ok=True)
os.makedirs(CFG.FINAL_CKPT_DIR, exist_ok=True)

if torch.cuda.is_available():
    if FORCE_SINGLE_GPU:
        DEVICE = torch.device("cuda:0")
        torch.cuda.set_device(0)
    else:
        DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_DP = False
AMP_ENABLED = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG.SEED)

print("=" * 80)
print("AISEHack Deadline-Safe Single Notebook")
print(f"Device           : {DEVICE}")
print(f"GPUs detected    : {N_GPUS}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU        : {props.name} ({props.total_memory / 1e9:.1f} GB)")
print("DataParallel     : OFF")
print(f"AMP              : {'ON' if AMP_ENABLED else 'OFF'}")
print(f"DEBUG            : {DEBUG}")
print(f"VAL windows/mo   : {VAL_WINDOWS_PER_MONTH}")
print(f"Top-K checkpoints: {TOPK_CKPTS}")
print(f"Final seeds      : {CFG.FINAL_SEEDS}")
print("=" * 80)


def month_length(month: str) -> int:
    path = os.path.join(CFG.RAW_ROOT, month, "cpm25.npy")
    return int(np.load(path, mmap_mode="r").shape[0])


def build_train_val_items():
    train_items = []
    val_items = []
    train_cutoffs = {}
    window_counts = {}

    for month in CFG.MONTHS:
        total_steps = month_length(month)
        starts = list(range(0, total_steps - CFG.HORIZON + 1, CFG.STRIDE))
        if len(starts) <= VAL_WINDOWS_PER_MONTH:
            raise ValueError(f"{month} has only {len(starts)} windows; cannot hold out {VAL_WINDOWS_PER_MONTH}")

        train_starts = starts[:-VAL_WINDOWS_PER_MONTH]
        val_starts = starts[-VAL_WINDOWS_PER_MONTH:]
        train_cutoffs[month] = total_steps - VAL_WINDOWS_PER_MONTH
        window_counts[month] = len(starts)

        for start in train_starts:
            train_items.append({"month": month, "start": start})
        for start in val_starts:
            val_items.append({"month": month, "start": start})

        print(
            f"{month}: total_windows={len(starts)} | "
            f"train_windows={len(train_starts)} | val_windows={len(val_starts)}"
        )

    return train_items, val_items, train_cutoffs, window_counts


TRAIN_ITEMS, VAL_ITEMS, TRAIN_CUTOFFS, WINDOW_COUNTS = build_train_val_items()
ALL_ITEMS = TRAIN_ITEMS + VAL_ITEMS

if DEBUG:
    TRAIN_ITEMS = TRAIN_ITEMS[: max(CFG.TRAIN_BATCH * CFG.GRAD_ACCUM * 4, 16)]
    VAL_ITEMS = VAL_ITEMS[: max(CFG.TRAIN_BATCH * 2, 8)]
    ALL_ITEMS = TRAIN_ITEMS + VAL_ITEMS


def load_lat_lon() -> tuple[np.ndarray, np.ndarray]:
    arr = np.load(CFG.LAT_LON_PATH).astype(np.float32)
    if arr.shape == (2, CFG.H, CFG.W):
        lat, lon = arr[0], arr[1]
    elif arr.shape == (CFG.H, CFG.W, 2):
        lat, lon = arr[..., 0], arr[..., 1]
    else:
        raise ValueError(f"Unexpected lat_long.npy shape: {arr.shape}")

    lat = (lat - lat.min()) / (lat.max() - lat.min() + CFG.EPS)
    lon = (lon - lon.min()) / (lon.max() - lon.min() + CFG.EPS)
    return lat.astype(np.float32), lon.astype(np.float32)


LAT_GRID, LON_GRID = load_lat_lon()


AISEHack Deadline-Safe Single Notebook
Device           : cuda:0
GPUs detected    : 2
Using GPU        : Tesla T4 (15.6 GB)
DataParallel     : OFF
AMP              : ON
DEBUG            : False
VAL windows/mo   : 48
Top-K checkpoints: 3
Final seeds      : [42, 52]
APRIL_16: total_windows=690 | train_windows=642 | val_windows=48
JULY_16: total_windows=714 | train_windows=666 | val_windows=48
OCT_16: total_windows=714 | train_windows=666 | val_windows=48
DEC_16: total_windows=714 | train_windows=666 | val_windows=48


In [3]:
# Cell 2

def load_month_base_arrays(month: str) -> dict[str, np.ndarray]:
    arrays = {}
    for feature in CFG.BASE_FEATURES:
        path = os.path.join(CFG.RAW_ROOT, month, f"{feature}.npy")
        arrays[feature] = np.load(path).astype(np.float32)
    return arrays


def load_test_base_arrays_memmap() -> dict[str, np.memmap]:
    arrays = {}
    for feature in CFG.BASE_FEATURES:
        path = os.path.join(CFG.TEST_ROOT, f"{feature}.npy")
        arrays[feature] = np.load(path, mmap_mode="r")
    return arrays


def compute_derived_features(base_arrays: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    u10 = base_arrays["u10"]
    v10 = base_arrays["v10"]
    pblh = base_arrays["pblh"]

    wind_speed = np.sqrt(np.maximum(u10 * u10 + v10 * v10, 0.0)).astype(np.float32)
    dudx = np.gradient(u10, axis=-1).astype(np.float32)
    dvdy = np.gradient(v10, axis=-2).astype(np.float32)
    wind_divergence = (dudx + dvdy).astype(np.float32)
    ventilation_index = (wind_speed * np.maximum(pblh, 0.0)).astype(np.float32)
    pblh_inverse = (1.0 / np.maximum(pblh, 1.0)).astype(np.float32)

    return {
        "wind_speed": wind_speed,
        "wind_divergence": wind_divergence,
        "ventilation_index": ventilation_index,
        "pblh_inverse": pblh_inverse,
    }


def compute_stats(train_cutoffs: dict[str, int] | None = None) -> dict[str, dict[str, float]]:
    accum = {
        feature: {"sum": 0.0, "sum_sq": 0.0, "count": 0}
        for feature in CFG.DYNAMIC_FEATURES
    }

    for month in tqdm(CFG.MONTHS, desc="Computing train-only stats"):
        base_arrays = load_month_base_arrays(month)
        all_arrays = {**base_arrays, **compute_derived_features(base_arrays)}
        for feature in CFG.DYNAMIC_FEATURES:
            arr = all_arrays[feature] if train_cutoffs is None else all_arrays[feature][: train_cutoffs[month]]
            accum[feature]["sum"] += float(arr.sum(dtype=np.float64))
            accum[feature]["sum_sq"] += float(np.square(arr, dtype=np.float64).sum(dtype=np.float64))
            accum[feature]["count"] += int(arr.size)
        del base_arrays, all_arrays
        gc.collect()

    stats = {}
    for feature, values in accum.items():
        mean = values["sum"] / values["count"]
        var = max(values["sum_sq"] / values["count"] - mean * mean, 1e-8)
        stats[feature] = {
            "mean": float(mean),
            "std": float(math.sqrt(var) + CFG.EPS),
        }
    return stats


def normalize_feature(arr: np.ndarray, feature: str, stats: dict[str, dict[str, float]]) -> np.ndarray:
    out = (arr.astype(np.float32) - stats[feature]["mean"]) / stats[feature]["std"]
    if feature in CFG.WIND_TANH_FEATURES:
        out = np.tanh(out)
    if feature in CFG.EMISSION_FEATURES:
        out = np.clip(out, -5.0, 5.0)
    return out.astype(np.float32, copy=False)


def build_month_cache(stats: dict[str, dict[str, float]]) -> dict[str, np.ndarray]:
    cache = {}
    total_gb = 0.0
    for month in CFG.MONTHS:
        base_arrays = load_month_base_arrays(month)
        all_arrays = {**base_arrays, **compute_derived_features(base_arrays)}
        stacked = [normalize_feature(all_arrays[feature], feature, stats) for feature in CFG.DYNAMIC_FEATURES]
        tensor = np.stack(stacked, axis=-1).astype(np.float32, copy=False)
        cache[month] = np.ascontiguousarray(tensor)
        total_gb += cache[month].nbytes / 1e9
        print(f"Cached {month}: {cache[month].shape} | {cache[month].nbytes / 1e9:.2f} GB")
        del base_arrays, all_arrays, stacked, tensor
        gc.collect()
    print(f"Cache RAM footprint: {total_gb:.2f} GB")
    return cache


def denorm_cpm25_np(x: np.ndarray, stats: dict[str, dict[str, float]]) -> np.ndarray:
    return x * stats["cpm25"]["std"] + stats["cpm25"]["mean"]


def denorm_cpm25_torch(x: torch.Tensor, stats: dict[str, dict[str, float]]) -> torch.Tensor:
    return x * stats["cpm25"]["std"] + stats["cpm25"]["mean"]


def build_hotspot_prior(
    month_cache: dict[str, np.ndarray],
    train_items: list[dict],
    stats: dict[str, dict[str, float]],
) -> np.ndarray:
    accum = np.zeros((CFG.H, CFG.W), dtype=np.float64)
    total_frames = 0
    items = train_items if not DEBUG else train_items[: min(len(train_items), 128)]

    for item in tqdm(items, desc="Building hotspot prior"):
        tensor = month_cache[item["month"]][:, :, :, CFG.TARGET_IDX]
        start = item["start"]
        window = tensor[start : start + CFG.HORIZON]
        input_seq = denorm_cpm25_np(window[: CFG.TIME_IN], stats)
        target_seq = denorm_cpm25_np(window[CFG.TIME_IN :], stats)

        baseline = input_seq.mean(axis=0)
        spread = input_seq.std(axis=0)
        threshold = baseline + CFG.EPISODE_THRESHOLD_STD * spread
        mask = target_seq > threshold[None, :, :]
        accum += mask.sum(axis=0, dtype=np.int64)
        total_frames += mask.shape[0]

    prior = accum / max(total_frames, 1)
    prior_t = torch.from_numpy(prior.astype(np.float32))[None, None]
    prior_t = F.avg_pool2d(
        prior_t,
        kernel_size=CFG.HOTSPOT_KERNEL,
        stride=1,
        padding=CFG.HOTSPOT_KERNEL // 2,
    )
    prior = prior_t.squeeze(0).squeeze(0).numpy()
    prior = (prior - prior.min()) / (prior.max() - prior.min() + CFG.EPS)
    return prior.astype(np.float32)


In [4]:
# Cell 3

class WindowDataset(Dataset):
    def __init__(self, month_cache: dict[str, np.ndarray], items: list[dict], hotspot_prior: np.ndarray):
        self.month_cache = month_cache
        self.items = items
        self.hotspot_prior = hotspot_prior.astype(np.float32)

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        item = self.items[idx]
        tensor = self.month_cache[item["month"]]
        start = item["start"]
        window = tensor[start : start + CFG.HORIZON]

        x_dyn = np.ascontiguousarray(window[: CFG.TIME_IN], dtype=np.float32)
        y = np.ascontiguousarray(window[CFG.TIME_IN :, :, :, CFG.TARGET_IDX].transpose(1, 2, 0))
        input_cpm25_seq = np.ascontiguousarray(x_dyn[:, :, :, CFG.TARGET_IDX], dtype=np.float32)
        last_cpm25 = np.ascontiguousarray(input_cpm25_seq[-1:, :, :].transpose(1, 2, 0))

        cpm25_mean_10 = input_cpm25_seq.mean(axis=0).astype(np.float32)
        cpm25_trend_3 = ((input_cpm25_seq[-1] - input_cpm25_seq[-4]) / 3.0).astype(np.float32)
        cpm25_std_10 = input_cpm25_seq.std(axis=0).astype(np.float32)
        x_static = np.stack(
            [
                LAT_GRID,
                LON_GRID,
                cpm25_mean_10,
                cpm25_trend_3,
                cpm25_std_10,
                self.hotspot_prior,
            ],
            axis=-1,
        ).astype(np.float32)

        return (
            torch.from_numpy(x_dyn),
            torch.from_numpy(x_static),
            torch.from_numpy(y),
            torch.from_numpy(last_cpm25),
            torch.from_numpy(input_cpm25_seq),
        )


class TestStreamingDataset(Dataset):
    def __init__(self, stats: dict[str, dict[str, float]], hotspot_prior: np.ndarray):
        self.stats = stats
        self.hotspot_prior = hotspot_prior.astype(np.float32)
        self.arrs = load_test_base_arrays_memmap()
        self.length = int(next(iter(self.arrs.values())).shape[0])

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int):
        base = {feature: self.arrs[feature][idx].astype(np.float32) for feature in CFG.BASE_FEATURES}
        derived = compute_derived_features(base)
        all_features = {**base, **derived}

        x_dyn = np.stack(
            [normalize_feature(all_features[feature], feature, self.stats) for feature in CFG.DYNAMIC_FEATURES],
            axis=-1,
        ).astype(np.float32)

        input_cpm25_seq = x_dyn[:, :, :, CFG.TARGET_IDX]
        last_cpm25 = input_cpm25_seq[-1:, :, :].transpose(1, 2, 0).astype(np.float32)
        cpm25_mean_10 = input_cpm25_seq.mean(axis=0).astype(np.float32)
        cpm25_trend_3 = ((input_cpm25_seq[-1] - input_cpm25_seq[-4]) / 3.0).astype(np.float32)
        cpm25_std_10 = input_cpm25_seq.std(axis=0).astype(np.float32)
        x_static = np.stack(
            [
                LAT_GRID,
                LON_GRID,
                cpm25_mean_10,
                cpm25_trend_3,
                cpm25_std_10,
                self.hotspot_prior,
            ],
            axis=-1,
        ).astype(np.float32)

        return (
            torch.from_numpy(np.ascontiguousarray(x_dyn)),
            torch.from_numpy(np.ascontiguousarray(x_static)),
            torch.from_numpy(np.ascontiguousarray(last_cpm25)),
            torch.tensor(idx, dtype=torch.long),
        )


def pad_tensor_2d(x: torch.Tensor) -> torch.Tensor:
    pad_h = CFG.PAD_H - x.shape[-2]
    pad_w = CFG.PAD_W - x.shape[-1]
    return F.pad(x, (0, pad_w, 0, pad_h))


def crop_tensor_2d(x: torch.Tensor) -> torch.Tensor:
    return x[..., : CFG.H, : CFG.W]


def build_abs_prediction(delta_pred: torch.Tensor, last_cpm25: torch.Tensor) -> torch.Tensor:
    return delta_pred + last_cpm25.expand(-1, -1, -1, CFG.TIME_OUT)


def build_episode_mask(
    input_cpm25_seq_norm: torch.Tensor,
    y_true_norm: torch.Tensor,
    stats: dict[str, dict[str, float]],
) -> torch.Tensor:
    input_denorm = denorm_cpm25_torch(input_cpm25_seq_norm, stats)
    y_denorm = denorm_cpm25_torch(y_true_norm, stats)
    baseline = input_denorm.mean(dim=1)
    spread = input_denorm.std(dim=1, correction=0)
    threshold = baseline.unsqueeze(-1) + CFG.EPISODE_THRESHOLD_STD * spread.unsqueeze(-1)
    return y_denorm > threshold


def smape_tensor(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    denom = (0.5 * (pred.abs() + target.abs())).clamp_min(eps)
    return (pred - target).abs() / denom


def compute_loss(
    delta_pred: torch.Tensor,
    y_true_norm: torch.Tensor,
    last_cpm25_norm: torch.Tensor,
    input_cpm25_seq_norm: torch.Tensor,
    x_static: torch.Tensor,
    stats: dict[str, dict[str, float]],
):
    abs_pred_norm = build_abs_prediction(delta_pred, last_cpm25_norm)
    delta_true_norm = y_true_norm - last_cpm25_norm.expand(-1, -1, -1, CFG.TIME_OUT)

    pred_denorm = denorm_cpm25_torch(abs_pred_norm, stats)
    true_denorm = denorm_cpm25_torch(y_true_norm, stats)
    episode_mask = build_episode_mask(input_cpm25_seq_norm, y_true_norm, stats)

    global_smape = smape_tensor(pred_denorm, true_denorm).mean()
    if bool(episode_mask.any()):
        episode_smape = smape_tensor(pred_denorm, true_denorm)[episode_mask].mean()
        episode_under = (true_denorm - pred_denorm).clamp_min(0.0)[episode_mask].mean()
    else:
        episode_smape = global_smape
        episode_under = pred_denorm.new_tensor(0.0)

    hotspot_prior = x_static[..., -1].unsqueeze(-1)
    hotspot_weight = 1.0 + CFG.HOTSPOT_WEIGHT_SCALE * hotspot_prior
    hotspot_l1 = (hotspot_weight * (pred_denorm - true_denorm).abs()).mean()
    delta_loss = F.huber_loss(delta_pred, delta_true_norm, delta=1.0)

    total = (
        CFG.LOSS_WEIGHTS["delta"] * delta_loss
        + CFG.LOSS_WEIGHTS["global_smape"] * global_smape
        + CFG.LOSS_WEIGHTS["episode_smape"] * episode_smape
        + CFG.LOSS_WEIGHTS["episode_under"] * episode_under
        + CFG.LOSS_WEIGHTS["hotspot"] * hotspot_l1
    )

    parts = {
        "delta": float(delta_loss.detach().item()),
        "global_smape": float(global_smape.detach().item()),
        "episode_smape": float(episode_smape.detach().item()),
        "episode_under": float(episode_under.detach().item()),
        "hotspot": float(hotspot_l1.detach().item()),
    }
    return total, abs_pred_norm, episode_mask, parts


class MetricTracker:
    def __init__(self, time_out: int):
        self.time_out = time_out
        self.global_smape_sum = 0.0
        self.global_count = 0
        self.episode_smape_sum = 0.0
        self.episode_count = 0
        self.corr_n = np.zeros(time_out, dtype=np.float64)
        self.sum_pred = np.zeros(time_out, dtype=np.float64)
        self.sum_true = np.zeros(time_out, dtype=np.float64)
        self.sum_pred_sq = np.zeros(time_out, dtype=np.float64)
        self.sum_true_sq = np.zeros(time_out, dtype=np.float64)
        self.sum_pred_true = np.zeros(time_out, dtype=np.float64)

    def update(self, pred: np.ndarray, true: np.ndarray, episode_mask: np.ndarray) -> None:
        smape = np.abs(pred - true) / (0.5 * (np.abs(pred) + np.abs(true)) + 1e-3)
        self.global_smape_sum += float(smape.sum(dtype=np.float64))
        self.global_count += smape.size

        if episode_mask.any():
            episode_values = smape[episode_mask]
            self.episode_smape_sum += float(episode_values.sum(dtype=np.float64))
            self.episode_count += int(episode_values.size)

        for t in range(self.time_out):
            mask_t = episode_mask[..., t]
            if not np.any(mask_t):
                continue
            pred_t = pred[..., t][mask_t].astype(np.float64, copy=False)
            true_t = true[..., t][mask_t].astype(np.float64, copy=False)
            n = pred_t.size
            self.corr_n[t] += n
            self.sum_pred[t] += pred_t.sum()
            self.sum_true[t] += true_t.sum()
            self.sum_pred_sq[t] += np.square(pred_t).sum()
            self.sum_true_sq[t] += np.square(true_t).sum()
            self.sum_pred_true[t] += (pred_t * true_t).sum()

    def compute(self) -> dict[str, float]:
        global_smape = self.global_smape_sum / max(self.global_count, 1)
        if self.episode_count > 0:
            episode_smape = self.episode_smape_sum / self.episode_count
        else:
            episode_smape = global_smape

        corrs = []
        for t in range(self.time_out):
            n = self.corr_n[t]
            if n < 2:
                continue
            cov = self.sum_pred_true[t] - (self.sum_pred[t] * self.sum_true[t] / n)
            var_pred = self.sum_pred_sq[t] - (self.sum_pred[t] ** 2 / n)
            var_true = self.sum_true_sq[t] - (self.sum_true[t] ** 2 / n)
            denom = math.sqrt(max(var_pred, 0.0) * max(var_true, 0.0))
            corr = cov / (denom + 1e-12) if denom > 0 else 0.0
            corrs.append(float(np.clip(corr, -1.0, 1.0)))

        episode_corr = float(np.mean(corrs)) if corrs else 0.0
        norm_global = float(np.clip(1.0 - global_smape / 2.0, 0.0, 1.0))
        norm_episode = float(np.clip(1.0 - episode_smape / 2.0, 0.0, 1.0))
        norm_corr = float(np.clip((episode_corr + 1.0) / 2.0, 0.0, 1.0))
        score_proxy = float(np.mean([norm_global, norm_episode, norm_corr]))

        return {
            "global_smape": float(global_smape),
            "episode_smape": float(episode_smape),
            "episode_corr": float(episode_corr),
            "score_proxy": float(score_proxy),
        }


In [5]:
# Cell 4

class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        groups = min(8, out_ch)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, hidden, kernel_size=1)
        self.fc2 = nn.Conv2d(hidden, channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.pool(x)
        scale = F.gelu(self.fc1(scale))
        scale = torch.sigmoid(self.fc2(scale))
        return x * scale


class DecoderSEBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        self.se = SEBlock(out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.se(self.conv(x))


class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch: int, hidden_ch: int):
        super().__init__()
        self.hidden_ch = hidden_ch
        self.gates = nn.Conv2d(in_ch + hidden_ch, 4 * hidden_ch, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor, h: torch.Tensor, c: torch.Tensor):
        gates = self.gates(torch.cat([x, h], dim=1))
        i, f, o, g = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c

    def init_state(self, batch: int, height: int, width: int, device: torch.device):
        shape = (batch, self.hidden_ch, height, width)
        return torch.zeros(shape, device=device), torch.zeros(shape, device=device)


class StackedConvLSTM(nn.Module):
    def __init__(self, in_ch: int, hidden_ch: int, num_layers: int = 2):
        super().__init__()
        self.layers = nn.ModuleList(
            [ConvLSTMCell(in_ch if idx == 0 else hidden_ch, hidden_ch) for idx in range(num_layers)]
        )

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        batch, steps, _, height, width = seq.shape
        states = [layer.init_state(batch, height, width, seq.device) for layer in self.layers]
        for t in range(steps):
            x_t = seq[:, t]
            for layer_idx, layer in enumerate(self.layers):
                h, c = states[layer_idx]
                h, c = layer(x_t, h, c)
                states[layer_idx] = (h, c)
                x_t = h
        return states[-1][0]


class ResidualConvLSTMUNetV3(nn.Module):
    def __init__(self):
        super().__init__()
        in_ch = CFG.N_DYNAMIC + CFG.N_STATIC
        self.pool = nn.MaxPool2d(2)
        self.enc1 = DoubleConv(in_ch, CFG.MODEL_BASE)
        self.enc2 = DoubleConv(CFG.MODEL_BASE, CFG.MODEL_BASE * 2)
        self.enc3 = DoubleConv(CFG.MODEL_BASE * 2, CFG.MODEL_HIDDEN)
        self.bottleneck_in = DoubleConv(CFG.MODEL_HIDDEN, CFG.MODEL_HIDDEN)
        self.temporal = StackedConvLSTM(CFG.MODEL_HIDDEN, CFG.MODEL_HIDDEN, num_layers=2)
        self.spatial_attention = nn.Conv2d(CFG.MODEL_HIDDEN, 1, kernel_size=1)
        self.dropout = nn.Dropout2d(CFG.DROPOUT)
        self.up3 = nn.ConvTranspose2d(CFG.MODEL_HIDDEN, CFG.MODEL_HIDDEN, kernel_size=2, stride=2)
        self.dec3 = DecoderSEBlock(CFG.MODEL_HIDDEN * 2, CFG.MODEL_HIDDEN)
        self.up2 = nn.ConvTranspose2d(CFG.MODEL_HIDDEN, CFG.MODEL_BASE * 2, kernel_size=2, stride=2)
        self.dec2 = DecoderSEBlock(CFG.MODEL_BASE * 4, CFG.MODEL_BASE * 2)
        self.up1 = nn.ConvTranspose2d(CFG.MODEL_BASE * 2, CFG.MODEL_BASE, kernel_size=2, stride=2)
        self.dec1 = DecoderSEBlock(CFG.MODEL_BASE * 2, CFG.MODEL_BASE)
        self.head = nn.Conv2d(CFG.MODEL_BASE, CFG.TIME_OUT, kernel_size=1)

    def encode_step(self, x_step: torch.Tensor):
        e1 = self.enc1(x_step)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        bottleneck = self.bottleneck_in(self.pool(e3))
        return e1, e2, e3, bottleneck

    def forward(self, x_dyn: torch.Tensor, x_static: torch.Tensor) -> torch.Tensor:
        x_dyn = x_dyn.permute(0, 1, 4, 2, 3).contiguous()
        x_static = x_static.permute(0, 3, 1, 2).contiguous()

        sequence = []
        skip1 = skip2 = skip3 = None
        for t in range(x_dyn.shape[1]):
            x_step = torch.cat([x_dyn[:, t], x_static], dim=1)
            x_step = pad_tensor_2d(x_step)
            skip1, skip2, skip3, bottleneck = self.encode_step(x_step)
            sequence.append(bottleneck.unsqueeze(1))

        context = self.temporal(torch.cat(sequence, dim=1))
        attention = torch.sigmoid(self.spatial_attention(context))
        context = context * (1.0 + attention)
        context = self.dropout(context)

        d3 = self.up3(context)
        if d3.shape[-2:] != skip3.shape[-2:]:
            d3 = F.interpolate(d3, size=skip3.shape[-2:], mode="bilinear", align_corners=False)
        d3 = self.dec3(torch.cat([d3, skip3], dim=1))

        d2 = self.up2(d3)
        if d2.shape[-2:] != skip2.shape[-2:]:
            d2 = F.interpolate(d2, size=skip2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([d2, skip2], dim=1))

        d1 = self.up1(d2)
        if d1.shape[-2:] != skip1.shape[-2:]:
            d1 = F.interpolate(d1, size=skip1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([d1, skip1], dim=1))

        out = crop_tensor_2d(self.head(d1))
        return out.permute(0, 2, 3, 1).contiguous()


def build_model() -> nn.Module:
    return ResidualConvLSTMUNetV3().to(DEVICE)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def autocast_context():
    if AMP_ENABLED:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def build_grad_scaler():
    if AMP_ENABLED and hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda", enabled=True)
        except TypeError:
            try:
                return torch.amp.GradScaler(enabled=True)
            except TypeError:
                pass
    return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = CFG.EMA_DECAY):
        self.decay = decay
        self.ema = deepcopy(model).to(DEVICE)
        self.ema.eval()
        for param in self.ema.parameters():
            param.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        src = model.state_dict()
        tgt = self.ema.state_dict()
        for key, value in tgt.items():
            src_value = src[key].detach()
            if torch.is_floating_point(value):
                value.mul_(self.decay).add_(src_value, alpha=1.0 - self.decay)
            else:
                value.copy_(src_value)


In [6]:
# Cell 5

def make_scheduler(optimizer: torch.optim.Optimizer, total_epochs: int, warmup_epochs: int):
    def lr_lambda(epoch: int) -> float:
        if total_epochs <= 1:
            return 1.0
        if epoch < warmup_epochs:
            return float(epoch + 1) / max(warmup_epochs, 1)
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs - 1, 1)
        progress = min(max(progress, 0.0), 1.0)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def run_shape_sanity(model: nn.Module, loader: DataLoader) -> None:
    x_dyn, x_static, y, last_cpm25, _ = next(iter(loader))
    x_dyn = x_dyn.to(DEVICE, non_blocking=True)
    x_static = x_static.to(DEVICE, non_blocking=True)
    y = y.to(DEVICE, non_blocking=True)
    last_cpm25 = last_cpm25.to(DEVICE, non_blocking=True)
    with torch.no_grad():
        with autocast_context():
            delta_pred = model(x_dyn, x_static)
            abs_pred = build_abs_prediction(delta_pred, last_cpm25)
    assert delta_pred.shape == (x_dyn.shape[0], CFG.H, CFG.W, CFG.TIME_OUT), delta_pred.shape
    assert abs_pred.shape == y.shape, (abs_pred.shape, y.shape)
    print("Shape sanity check passed")


def validate_epoch(model: nn.Module, loader: DataLoader, stats: dict[str, dict[str, float]], limit_batches: int | None = None):
    model.eval()
    tracker = MetricTracker(CFG.TIME_OUT)
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch_idx, (x_dyn, x_static, y, last_cpm25, input_cpm25_seq) in enumerate(loader):
            if limit_batches is not None and batch_idx >= limit_batches:
                break

            x_dyn = x_dyn.to(DEVICE, non_blocking=True)
            x_static = x_static.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            last_cpm25 = last_cpm25.to(DEVICE, non_blocking=True)
            input_cpm25_seq = input_cpm25_seq.to(DEVICE, non_blocking=True)

            with autocast_context():
                delta_pred = model(x_dyn, x_static)
                loss, abs_pred, episode_mask, _ = compute_loss(
                    delta_pred,
                    y,
                    last_cpm25,
                    input_cpm25_seq,
                    x_static,
                    stats,
                )

            total_loss += float(loss.detach().item())
            total_batches += 1

            pred_denorm = denorm_cpm25_torch(abs_pred.float(), stats).cpu().numpy()
            true_denorm = denorm_cpm25_torch(y.float(), stats).cpu().numpy()
            tracker.update(pred_denorm, true_denorm, episode_mask.cpu().numpy().astype(bool))

    metrics = tracker.compute()
    metrics["val_loss"] = total_loss / max(total_batches, 1)
    return metrics


def cleanup_checkpoint_dir(checkpoint_dir: str) -> None:
    for name in os.listdir(checkpoint_dir):
        if name.endswith(".pt"):
            path = os.path.join(checkpoint_dir, name)
            try:
                os.remove(path)
            except OSError:
                pass


def save_topk_aliases(topk_entries: list[dict], checkpoint_dir: str) -> list[str]:
    final_paths = []
    for rank, entry in enumerate(topk_entries, start=1):
        target = os.path.join(checkpoint_dir, f"top{rank}.pt")
        shutil.copyfile(entry["path"], target)
        final_paths.append(target)
    return final_paths


In [7]:
# Cell 6

def train_model(
    train_dataset: Dataset,
    val_dataset: Dataset,
    stats: dict[str, dict[str, float]],
    checkpoint_dir: str,
    log_path: str,
    seed: int,
    epochs: int,
    patience: int | None,
) -> dict:
    set_seed(seed)
    cleanup_checkpoint_dir(checkpoint_dir)

    train_loader = make_loader(train_dataset, CFG.TRAIN_BATCH, shuffle=True)
    val_loader = make_loader(val_dataset, CFG.TRAIN_BATCH, shuffle=False)

    model = build_model()
    ema = ModelEMA(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = make_scheduler(optimizer, epochs if not DEBUG else 1, CFG.WARMUP_EPOCHS)
    scaler = build_grad_scaler()
    run_shape_sanity(model, train_loader)

    best_score = -float("inf")
    best_epoch = 0
    stale_epochs = 0
    history = []
    topk_entries = []
    debug_train_limit = 2 if DEBUG else None
    debug_val_limit = 2 if DEBUG else None

    print(f"Trainable parameters: {count_parameters(model) / 1e6:.2f}M")

    total_epochs = 1 if DEBUG else epochs
    for epoch in range(total_epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_start = time.time()
        total_loss = 0.0
        total_batches = 0
        oom_skips = 0

        limit_steps = min(len(train_loader), debug_train_limit) if debug_train_limit is not None else len(train_loader)

        for batch_idx, (x_dyn, x_static, y, last_cpm25, input_cpm25_seq) in enumerate(train_loader):
            if debug_train_limit is not None and batch_idx >= debug_train_limit:
                break

            x_dyn = x_dyn.to(DEVICE, non_blocking=True)
            x_static = x_static.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            last_cpm25 = last_cpm25.to(DEVICE, non_blocking=True)
            input_cpm25_seq = input_cpm25_seq.to(DEVICE, non_blocking=True)

            try:
                with autocast_context():
                    delta_pred = model(x_dyn, x_static)
                    loss, _, _, _ = compute_loss(
                        delta_pred,
                        y,
                        last_cpm25,
                        input_cpm25_seq,
                        x_static,
                        stats,
                    )
                    scaled_loss = loss / CFG.GRAD_ACCUM

                if AMP_ENABLED:
                    scaler.scale(scaled_loss).backward()
                else:
                    scaled_loss.backward()

                should_step = ((batch_idx + 1) % CFG.GRAD_ACCUM == 0) or ((batch_idx + 1) == limit_steps)
                if should_step:
                    if AMP_ENABLED:
                        scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    if AMP_ENABLED:
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    ema.update(model)

                total_loss += float(loss.detach().item())
                total_batches += 1
            except RuntimeError as exc:
                if "out of memory" not in str(exc).lower():
                    raise
                oom_skips += 1
                optimizer.zero_grad(set_to_none=True)
                gc.collect()
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
                print(f"[train] skipped OOM batch {batch_idx}")

        scheduler.step()

        val_metrics = validate_epoch(ema.ema, val_loader, stats, limit_batches=debug_val_limit)
        epoch_log = {
            "epoch": epoch + 1,
            "train_loss": round(total_loss / max(total_batches, 1), 6),
            "lr": round(float(optimizer.param_groups[0]["lr"]), 8),
            "minutes": round((time.time() - epoch_start) / 60.0, 2),
            "oom_skips": oom_skips,
            "val_loss": round(val_metrics["val_loss"], 6),
            "global_smape": round(val_metrics["global_smape"], 6),
            "episode_smape": round(val_metrics["episode_smape"], 6),
            "episode_corr": round(val_metrics["episode_corr"], 6),
            "score_proxy": round(val_metrics["score_proxy"], 6),
        }
        history.append(epoch_log)
        print(f"[deadline_run] epoch {epoch + 1:02d} | {epoch_log}")

        candidate_path = os.path.join(checkpoint_dir, f"epoch_{epoch + 1:02d}.pt")
        torch.save(
            {
                "epoch": epoch + 1,
                "score_proxy": val_metrics["score_proxy"],
                "model_state": ema.ema.state_dict(),
                "history": history,
            },
            candidate_path,
        )
        topk_entries.append(
            {
                "path": candidate_path,
                "score": float(val_metrics["score_proxy"]),
                "epoch": epoch + 1,
            }
        )
        topk_entries.sort(key=lambda item: item["score"], reverse=True)

        while len(topk_entries) > TOPK_CKPTS:
            removed = topk_entries.pop(-1)
            if os.path.exists(removed["path"]):
                os.remove(removed["path"])

        if val_metrics["score_proxy"] > best_score:
            best_score = float(val_metrics["score_proxy"])
            best_epoch = epoch + 1
            stale_epochs = 0
        else:
            stale_epochs += 1
            if patience is not None and stale_epochs >= patience:
                print(f"[deadline_run] early stopping at epoch {epoch + 1}")
                break

    with open(log_path, "w") as handle:
        json.dump(history, handle, indent=2)

    final_top_paths = save_topk_aliases(topk_entries, checkpoint_dir)
    del model, ema, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "best_score": best_score,
        "best_epoch": best_epoch,
        "top_paths": final_top_paths,
        "history": history,
    }


def train_final_ensemble(
    train_dataset: Dataset,
    stats: dict[str, dict[str, float]],
    epochs: int,
) -> list[str]:
    cleanup_checkpoint_dir(CFG.FINAL_CKPT_DIR)
    final_paths = []
    final_logs = []

    for seed in CFG.FINAL_SEEDS:
        set_seed(seed)
        train_loader = make_loader(train_dataset, CFG.TRAIN_BATCH, shuffle=True)
        model = build_model()
        ema = ModelEMA(model)
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
        scheduler = make_scheduler(optimizer, epochs if not DEBUG else 1, CFG.WARMUP_EPOCHS)
        scaler = build_grad_scaler()

        run_shape_sanity(model, train_loader)
        history = []
        total_epochs = 1 if DEBUG else epochs

        for epoch in range(total_epochs):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            epoch_start = time.time()
            total_loss = 0.0
            total_batches = 0
            oom_skips = 0
            debug_train_limit = 2 if DEBUG else None
            limit_steps = min(len(train_loader), debug_train_limit) if debug_train_limit is not None else len(train_loader)

            for batch_idx, (x_dyn, x_static, y, last_cpm25, input_cpm25_seq) in enumerate(train_loader):
                if debug_train_limit is not None and batch_idx >= debug_train_limit:
                    break

                x_dyn = x_dyn.to(DEVICE, non_blocking=True)
                x_static = x_static.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                last_cpm25 = last_cpm25.to(DEVICE, non_blocking=True)
                input_cpm25_seq = input_cpm25_seq.to(DEVICE, non_blocking=True)

                try:
                    with autocast_context():
                        delta_pred = model(x_dyn, x_static)
                        loss, _, _, _ = compute_loss(
                            delta_pred,
                            y,
                            last_cpm25,
                            input_cpm25_seq,
                            x_static,
                            stats,
                        )
                        scaled_loss = loss / CFG.GRAD_ACCUM

                    if AMP_ENABLED:
                        scaler.scale(scaled_loss).backward()
                    else:
                        scaled_loss.backward()

                    should_step = ((batch_idx + 1) % CFG.GRAD_ACCUM == 0) or ((batch_idx + 1) == limit_steps)
                    if should_step:
                        if AMP_ENABLED:
                            scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        if AMP_ENABLED:
                            scaler.step(optimizer)
                            scaler.update()
                        else:
                            optimizer.step()
                        optimizer.zero_grad(set_to_none=True)
                        ema.update(model)

                    total_loss += float(loss.detach().item())
                    total_batches += 1
                except RuntimeError as exc:
                    if "out of memory" not in str(exc).lower():
                        raise
                    oom_skips += 1
                    optimizer.zero_grad(set_to_none=True)
                    gc.collect()
                    if DEVICE.type == "cuda":
                        torch.cuda.empty_cache()
                    print(f"[final_seed_{seed}] skipped OOM batch {batch_idx}")

            scheduler.step()
            epoch_log = {
                "seed": seed,
                "epoch": epoch + 1,
                "train_loss": round(total_loss / max(total_batches, 1), 6),
                "lr": round(float(optimizer.param_groups[0]["lr"]), 8),
                "minutes": round((time.time() - epoch_start) / 60.0, 2),
                "oom_skips": oom_skips,
            }
            history.append(epoch_log)
            print(f"[final_seed_{seed}] epoch {epoch + 1:02d} | {epoch_log}")

        checkpoint_path = os.path.join(CFG.FINAL_CKPT_DIR, f"seed{seed}.pt")
        torch.save(
            {
                "seed": seed,
                "epoch": total_epochs,
                "model_state": ema.ema.state_dict(),
                "history": history,
            },
            checkpoint_path,
        )
        final_paths.append(checkpoint_path)
        final_logs.extend(history)

        del model, ema, optimizer, scheduler, scaler, train_loader
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    with open(CFG.FINAL_LOG_PATH, "w") as handle:
        json.dump(final_logs, handle, indent=2)

    return final_paths

In [8]:
# Cell 7

def load_eval_model(checkpoint_path: str) -> nn.Module:
    model = build_model()
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model


@torch.no_grad()
def infer_checkpoint(checkpoint_path: str, dataset: Dataset, stats: dict[str, dict[str, float]]) -> np.ndarray:
    loader = make_loader(dataset, CFG.INFER_BATCH, shuffle=False)
    model = load_eval_model(checkpoint_path)
    preds = np.zeros((len(dataset), CFG.H, CFG.W, CFG.TIME_OUT), dtype=np.float32)
    debug_limit = 1 if DEBUG else None

    for batch_idx, (x_dyn, x_static, last_cpm25, idx) in enumerate(
        tqdm(loader, desc=os.path.basename(checkpoint_path))
    ):
        if debug_limit is not None and batch_idx >= debug_limit:
            break

        x_dyn = x_dyn.to(DEVICE, non_blocking=True)
        x_static = x_static.to(DEVICE, non_blocking=True)
        last_cpm25 = last_cpm25.to(DEVICE, non_blocking=True)

        with autocast_context():
            delta_pred = model(x_dyn, x_static)
            abs_pred_norm = build_abs_prediction(delta_pred, last_cpm25)

        pred = denorm_cpm25_torch(abs_pred_norm.float(), stats).cpu().numpy()
        preds[idx.numpy()] = pred

    del model, loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return preds


def get_top_checkpoint_paths() -> list[str]:
    paths = []
    for rank in range(1, TOPK_CKPTS + 1):
        path = os.path.join(CFG.SELECT_CKPT_DIR, f"top{rank}.pt")
        if os.path.exists(path):
            paths.append(path)
    if not paths:
        raise FileNotFoundError("No top-k checkpoints found. Run training first.")
    return paths


def get_final_checkpoint_paths() -> list[str]:
    paths = []
    for seed in CFG.FINAL_SEEDS:
        path = os.path.join(CFG.FINAL_CKPT_DIR, f"seed{seed}.pt")
        if os.path.exists(path):
            paths.append(path)
    return paths


def run_inference(stats: dict[str, dict[str, float]], hotspot_prior: np.ndarray, checkpoint_paths: list[str]) -> None:
    dataset = TestStreamingDataset(stats, hotspot_prior)
    ensemble = np.zeros((len(dataset), CFG.H, CFG.W, CFG.TIME_OUT), dtype=np.float32)

    for checkpoint_path in checkpoint_paths:
        ensemble += infer_checkpoint(checkpoint_path, dataset, stats)

    ensemble /= max(len(checkpoint_paths), 1)
    ensemble = np.clip(ensemble, 0.0, None).astype(np.float32)

    if DEBUG:
        print(f"DEBUG inference complete: shape={ensemble.shape}")
    else:
        assert ensemble.shape == (218, CFG.H, CFG.W, CFG.TIME_OUT), ensemble.shape
        np.save(CFG.PREDS_PATH, ensemble)
        print(f"Saved predictions to {CFG.PREDS_PATH}")
        print(
            f"preds.npy -> shape={ensemble.shape}, dtype={ensemble.dtype}, "
            f"min={ensemble.min():.3f}, max={ensemble.max():.3f}, mean={ensemble.mean():.3f}"
        )


In [9]:
# Cell 8

def prepare_selection_artifacts():
    stats = compute_stats(TRAIN_CUTOFFS)
    with open(CFG.SELECT_STATS_PATH, "w") as handle:
        json.dump(stats, handle, indent=2)

    month_cache = build_month_cache(stats)
    hotspot_prior = build_hotspot_prior(month_cache, TRAIN_ITEMS, stats)
    np.save(CFG.SELECT_HOTSPOT_PATH, hotspot_prior.astype(np.float32))
    return stats, month_cache, hotspot_prior


def prepare_final_artifacts():
    stats = compute_stats(None)
    with open(CFG.STATS_PATH, "w") as handle:
        json.dump(stats, handle, indent=2)

    month_cache = build_month_cache(stats)
    hotspot_prior = build_hotspot_prior(month_cache, ALL_ITEMS, stats)
    np.save(CFG.HOTSPOT_PATH, hotspot_prior.astype(np.float32))
    return stats, month_cache, hotspot_prior


def load_saved_artifacts(use_final: bool = True):
    stats_path = CFG.STATS_PATH if use_final else CFG.SELECT_STATS_PATH
    hotspot_path = CFG.HOTSPOT_PATH if use_final else CFG.SELECT_HOTSPOT_PATH

    if not os.path.exists(stats_path):
        raise FileNotFoundError(f"Missing stats file: {stats_path}")
    if not os.path.exists(hotspot_path):
        raise FileNotFoundError(f"Missing hotspot prior: {hotspot_path}")
    with open(stats_path, "r") as handle:
        stats = json.load(handle)
    hotspot_prior = np.load(hotspot_path).astype(np.float32)
    return stats, hotspot_prior


def resolve_final_epochs(selection_result: dict | None) -> int:
    if selection_result is not None:
        return max(int(selection_result["best_epoch"]), 1)

    select_top1 = os.path.join(CFG.SELECT_CKPT_DIR, "top1.pt")
    if os.path.exists(select_top1):
        ckpt = torch.load(select_top1, map_location="cpu")
        epoch = int(ckpt.get("epoch", 0))
        if epoch > 0:
            return epoch

    return max(CFG.EPOCHS - 2, 1)


def main() -> None:
    selection_result = None

    if RUN_TRAIN:
        stats, month_cache, hotspot_prior = prepare_selection_artifacts()
        train_dataset = WindowDataset(month_cache, TRAIN_ITEMS, hotspot_prior)
        val_dataset = WindowDataset(month_cache, VAL_ITEMS, hotspot_prior)

        selection_result = train_model(
            train_dataset,
            val_dataset,
            stats,
            checkpoint_dir=CFG.SELECT_CKPT_DIR,
            log_path=CFG.SELECT_LOG_PATH,
            seed=CFG.SEED,
            epochs=CFG.EPOCHS,
            patience=CFG.PATIENCE,
        )
        print(
            f"Selection complete | best_epoch={selection_result['best_epoch']} | "
            f"best_score={selection_result['best_score']:.6f}"
        )

        del train_dataset, val_dataset, month_cache
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    if RUN_FINAL_TRAIN:
        final_epochs = resolve_final_epochs(selection_result)
        final_stats, final_month_cache, final_hotspot_prior = prepare_final_artifacts()
        final_dataset = WindowDataset(final_month_cache, ALL_ITEMS, final_hotspot_prior)
        final_paths = train_final_ensemble(final_dataset, final_stats, epochs=final_epochs)
        print(f"Final ensemble complete | epochs={final_epochs} | checkpoints={len(final_paths)}")

        del final_dataset, final_month_cache
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    if RUN_INFER:
        final_paths = get_final_checkpoint_paths()
        if final_paths:
            stats, hotspot_prior = load_saved_artifacts(use_final=True)
            checkpoint_paths = final_paths
        else:
            stats, hotspot_prior = load_saved_artifacts(use_final=False)
            checkpoint_paths = get_top_checkpoint_paths()
        run_inference(stats, hotspot_prior, checkpoint_paths)

    print("Deadline-safe notebook complete.")


if __name__ == "__main__":
    main()


Computing train-only stats:   0%|          | 0/4 [00:00<?, ?it/s]

Cached APRIL_16: (715, 140, 124, 20) | 0.99 GB
Cached JULY_16: (739, 140, 124, 20) | 1.03 GB
Cached OCT_16: (739, 140, 124, 20) | 1.03 GB
Cached DEC_16: (739, 140, 124, 20) | 1.03 GB
Cache RAM footprint: 4.07 GB


Building hotspot prior:   0%|          | 0/2640 [00:00<?, ?it/s]

Shape sanity check passed
Trainable parameters: 5.88M
[deadline_run] epoch 01 | {'epoch': 1, 'train_loss': 2.422905, 'lr': 0.0008, 'minutes': 2.87, 'oom_skips': 0, 'val_loss': 2.382516, 'global_smape': 0.286035, 'episode_smape': 0.214281, 'episode_corr': 0.945753, 'score_proxy': 0.907573}
[deadline_run] epoch 02 | {'epoch': 2, 'train_loss': 2.034387, 'lr': 0.00076955, 'minutes': 2.58, 'oom_skips': 0, 'val_loss': 2.086421, 'global_smape': 0.251061, 'episode_smape': 0.180948, 'episode_corr': 0.946845, 'score_proxy': 0.919139}
[deadline_run] epoch 03 | {'epoch': 3, 'train_loss': 1.862445, 'lr': 0.00068284, 'minutes': 2.57, 'oom_skips': 0, 'val_loss': 1.927341, 'global_smape': 0.235967, 'episode_smape': 0.168917, 'episode_corr': 0.951359, 'score_proxy': 0.924413}
[deadline_run] epoch 04 | {'epoch': 4, 'train_loss': 1.741524, 'lr': 0.00055307, 'minutes': 2.59, 'oom_skips': 0, 'val_loss': 1.822627, 'global_smape': 0.226217, 'episode_smape': 0.162992, 'episode_corr': 0.954919, 'score_proxy': 

Computing train-only stats:   0%|          | 0/4 [00:00<?, ?it/s]

...

Building hotspot prior:   0%|          | 0/2832 [00:00<?, ?it/s]

Shape sanity check passed
[final_seed_42] epoch 01 | {'seed': 42, 'epoch': 1, 'train_loss': 2.426409, 'lr': 0.0008, 'minutes': 2.67, 'oom_skips': 0}
[final_seed_42] epoch 02 | {'seed': 42, 'epoch': 2, 'train_loss': 2.033882, 'lr': 0.00072361, 'minutes': 2.66, 'oom_skips': 0}
[final_seed_42] epoch 03 | {'seed': 42, 'epoch': 3, 'train_loss': 1.865274, 'lr': 0.00052361, 'minutes': 2.65, 'oom_skips': 0}
[final_seed_42] epoch 04 | {'seed': 42, 'epoch': 4, 'train_loss': 1.720354, 'lr': 0.00027639, 'minutes': 2.65, 'oom_skips': 0}
[final_seed_42] epoch 05 | {'seed': 42, 'epoch': 5, 'train_loss': 1.594889, 'lr': 7.639e-05, 'minutes': 2.65, 'oom_skips': 0}
[final_seed_42] epoch 06 | {'seed': 42, 'epoch': 6, 'train_loss': 1.488397, 'lr': 0.0, 'minutes': 2.65, 'oom_skips': 0}
[final_seed_42] epoch 07 | {'seed': 42, 'epoch': 7, 'train_loss': 1.455935, 'lr': 0.0, 'minutes': 2.64, 'oom_skips': 0}
Shape sanity check passed
[final_seed_52] epoch 01 | {'seed': 52, 'epoch': 1, 'train_loss': 2.482796, 'l

seed42.pt:   0%|          | 0/55 [00:00<?, ?it/s]

seed52.pt:   0%|          | 0/55 [00:00<?, ?it/s]

Saved predictions to /kaggle/working/preds.npy
preds.npy -> shape=(218, 140, 124, 16), dtype=float32, min=0.000, max=1516.882, mean=36.386
Deadline-safe notebook complete.
